In [ ]:
%load_ext autoreload
%autoreload 2

# PPO with AAC

> PPO with AAC class
> 
>Title: RDPG for VEOS
>Author: Binjian Xin
>Date created: 2025/03/21
>Last modified: 2025/03/21
>Description: Adapted from Humanoid GYM
>
>

In [ ]:
#| default_exp agent.aac.aac

In [ ]:
#| export
import logging
import os
from dataclasses import dataclass
from pathlib import Path
from typing import Optional, Tuple
import numpy as np
import pandas as pd
import tensorflow as tf
from typeguard import check_type

In [ ]:
#| export
from tspace.agent.utils.hyperparams import HyperParamRDPG, HyperParamDDPG, HyperParamIDQL
from tspace.storage.buffer.dask import DaskBuffer
from tspace.storage.buffer.mongo import MongoBuffer  # type: ignore
from tspace.data.core import PoolQuery  # type: ignore
from tspace.data.time import veos_lifetime_end_date, veos_lifetime_start_date

In [ ]:
#| export
from tspace.agent.ppo import PPO # type: ignore
from tspace.agent.aac.actor_critic import ActorCritic # type: ignore
from tspace.agent.aac.rollout_storage import RolloutStorage # type: ignore

In [ ]:

#| hide
from tspace.config.robots import Robot, RobotInCloud, robots_by_id
from tspace.config.drivers import Driver
import logging
from typing import Union
from tspace.data.core import (
    RE_RECIPEKEY,
    ActionSpecs,
    ObservationMetaCloud,
    ObservationMetaECU,
    RewardSpecs,
    StateSpecsCloud,
    StateSpecsECU,
    get_filemeta_config,
)

In [ ]:
#| export
@dataclass
# class AAC(PPO):
class AAC:
    actor_critic: ActorCritic
    def __init__(self,
                 actor_critic,
                 num_learning_epochs=1,
                 num_mini_batches=1,
                 clip_param=0.2,
                 gamma=0.998,
                 lam=0.95,
                 value_loss_coef=1.0,
                 entropy_coef=0.0,
                 learning_rate=1e-3,
                 max_grad_norm=1.0,
                 use_clipped_value_loss=True,
                 schedule="fixed",
                 desired_kl=0.01,
                 device='cpu',
                 ):

        self.device = device

        self.desired_kl = desired_kl
        self.schedule = schedule
        self.learning_rate = learning_rate

        # PPO components
        self.actor_critic = actor_critic
        self.actor_critic.to(self.device)
        self.storage = None # initialized later
        self.optimizer = optim.Adam(self.actor_critic.parameters(), lr=learning_rate)
        self.transition = RolloutStorage.Transition()

        # PPO parameters
        self.clip_param = clip_param
        self.num_learning_epochs = num_learning_epochs
        self.num_mini_batches = num_mini_batches
        self.value_loss_coef = value_loss_coef
        self.entropy_coef = entropy_coef
        self.gamma = gamma
        self.lam = lam
        self.max_grad_norm = max_grad_norm
        self.use_clipped_value_loss = use_clipped_value_loss

    def init_storage(self, num_envs, num_transitions_per_env, actor_obs_shape, critic_obs_shape, action_shape):
        self.storage = RolloutStorage(num_envs, num_transitions_per_env, actor_obs_shape, critic_obs_shape, action_shape, self.device)

    def test_mode(self):
        self.actor_critic.test()
    
    def train_mode(self):
        self.actor_critic.train()

    def act(self, obs, critic_obs):
        # Compute the actions and values
        self.transition.actions = self.actor_critic.act(obs).detach()
        self.transition.values = self.actor_critic.evaluate(critic_obs).detach()
        self.transition.actions_log_prob = self.actor_critic.get_actions_log_prob(self.transition.actions).detach()
        self.transition.action_mean = self.actor_critic.action_mean.detach()
        self.transition.action_sigma = self.actor_critic.action_std.detach()
        # need to record obs and critic_obs before env.step()
        self.transition.observations = obs
        self.transition.critic_observations = critic_obs
        return self.transition.actions
    
    def process_env_step(self, rewards, dones, infos):
        self.transition.rewards = rewards.clone()
        self.transition.dones = dones
        # Bootstrapping on time outs
        if 'time_outs' in infos:
            self.transition.rewards += self.gamma * torch.squeeze(self.transition.values * infos['time_outs'].unsqueeze(1).to(self.device), 1)

        # Record the transition
        self.storage.add_transitions(self.transition)
        self.transition.clear()
        self.actor_critic.reset(dones)
    
    def compute_returns(self, last_critic_obs):
        last_values= self.actor_critic.evaluate(last_critic_obs).detach()
        self.storage.compute_returns(last_values, self.gamma, self.lam)

    def update(self):
        mean_value_loss = 0
        mean_surrogate_loss = 0

        generator = self.storage.mini_batch_generator(self.num_mini_batches, self.num_learning_epochs)
        for obs_batch, critic_obs_batch, actions_batch, target_values_batch, advantages_batch, returns_batch, old_actions_log_prob_batch, \
            old_mu_batch, old_sigma_batch, hid_states_batch, masks_batch in generator:


                self.actor_critic.act(obs_batch, masks=masks_batch, hidden_states=hid_states_batch[0])
                actions_log_prob_batch = self.actor_critic.get_actions_log_prob(actions_batch)
                value_batch = self.actor_critic.evaluate(critic_obs_batch, masks=masks_batch, hidden_states=hid_states_batch[1])
                mu_batch = self.actor_critic.action_mean
                sigma_batch = self.actor_critic.action_std
                entropy_batch = self.actor_critic.entropy

                # KL
                if self.desired_kl != None and self.schedule == 'adaptive':
                    with torch.inference_mode():
                        kl = torch.sum(
                            torch.log(sigma_batch / old_sigma_batch + 1.e-5) + (torch.square(old_sigma_batch) + torch.square(old_mu_batch - mu_batch)) / (2.0 * torch.square(sigma_batch)) - 0.5, axis=-1)
                        kl_mean = torch.mean(kl)

                        if kl_mean > self.desired_kl * 2.0:
                            self.learning_rate = max(1e-5, self.learning_rate / 1.5)
                        elif kl_mean < self.desired_kl / 2.0 and kl_mean > 0.0:
                            self.learning_rate = min(1e-2, self.learning_rate * 1.5)
                        
                        for param_group in self.optimizer.param_groups:
                            param_group['lr'] = self.learning_rate


                # Surrogate loss
                ratio = torch.exp(actions_log_prob_batch - torch.squeeze(old_actions_log_prob_batch))
                surrogate = -torch.squeeze(advantages_batch) * ratio
                surrogate_clipped = -torch.squeeze(advantages_batch) * torch.clamp(ratio, 1.0 - self.clip_param,
                                                                                1.0 + self.clip_param)
                surrogate_loss = torch.max(surrogate, surrogate_clipped).mean()

                # Value function loss
                if self.use_clipped_value_loss:
                    value_clipped = target_values_batch + (value_batch - target_values_batch).clamp(-self.clip_param,
                                                                                                    self.clip_param)
                    value_losses = (value_batch - returns_batch).pow(2)
                    value_losses_clipped = (value_clipped - returns_batch).pow(2)
                    value_loss = torch.max(value_losses, value_losses_clipped).mean()
                else:
                    value_loss = (returns_batch - value_batch).pow(2).mean()

                loss = surrogate_loss + self.value_loss_coef * value_loss - self.entropy_coef * entropy_batch.mean()

                # Gradient step
                self.optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(self.actor_critic.parameters(), self.max_grad_norm)
                self.optimizer.step()

                mean_value_loss += value_loss.item()
                mean_surrogate_loss += surrogate_loss.item()

        num_updates = self.num_learning_epochs * self.num_mini_batches
        mean_value_loss /= num_updates
        mean_surrogate_loss /= num_updates
        self.storage.clear()

        return mean_value_loss, mean_surrogate_loss


In [ ]:
#| hide
from nbdev.showdoc import show_doc

In [ ]:
show_doc(AAC.init_storage)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/aac.py#L83){target="_blank" style="float:right; font-size:smaller"}

### AAC.init_storage

>      AAC.init_storage (num_envs, num_transitions_per_env, actor_obs_shape,
>                        critic_obs_shape, action_shape)

In [ ]:
show_doc(AAC.test_mode)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/aac.py#L100){target="_blank" style="float:right; font-size:smaller"}

### AAC.test_mode

>      AAC.test_mode ()

In [ ]:
show_doc(AAC.train_mode)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/aac.py#L103){target="_blank" style="float:right; font-size:smaller"}

### AAC.train_mode

>      AAC.train_mode ()

In [ ]:
show_doc(AAC.train_mode)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/aac.py#L103){target="_blank" style="float:right; font-size:smaller"}

### AAC.train_mode

>      AAC.train_mode ()

In [ ]:
show_doc(AAC.act)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/aac.py#L106){target="_blank" style="float:right; font-size:smaller"}

### AAC.act

>      AAC.act (obs, critic_obs)

In [ ]:
show_doc(AAC.process_env_step)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/aac.py#L120){target="_blank" style="float:right; font-size:smaller"}

### AAC.process_env_step

>      AAC.process_env_step (rewards, dones, infos)

In [ ]:
show_doc(AAC.compute_returns)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/aac.py#L136){target="_blank" style="float:right; font-size:smaller"}

### AAC.compute_returns

>      AAC.compute_returns (last_critic_obs)

In [ ]:
show_doc(AAC.update)

---

[source](https://github.com/Binjian/tspace/blob/main/tspace/agent/aac/aac.py#L140){target="_blank" style="float:right; font-size:smaller"}

### AAC.update

>      AAC.update ()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()